# Operator Showcase

This notebook demonstrates all major operator categories with practical examples.

## Operator Categories
1. **Ingest Operators** - Load documents from various sources
2. **Extract Operators** - Extract text and entities from documents
3. **Chunking Operators** - Split documents into manageable pieces
4. **Quality Operators** - Assess and improve data quality
5. **Functional Operators** - Transform and enrich data
6. **Storage Operators** - Persist data for later use

## Prerequisites
- Sample documents in `sample_documents/` directory
- Ollama running (for some operators)
- Virtual environment activated

## Setup and Imports

In [ ]:
import sys
from pathlib import Path
from pprint import pprint
import pandas as pd

# Add src to path if needed
import os
if 'PYTHONPATH' not in os.environ:
    src_path = Path.cwd().parent.parent / "src"
    sys.path.insert(0, str(src_path))

from docpipe.lib.docpipe_flow_manager import DocpipeFlowManager

print("✓ Imports loaded successfully")

## List All Available Operators

First, let's see what operators are available:

In [ ]:
# List all available operators
operators = DocpipeFlowManager.list_operators(verbose=False)
print("\nOperators organized by category")

## 1. Ingest Operators

### IngestSource (filesystem) - Load files from local filesystem

In [ ]:
ingest_flow = {
    "flow_name": "ingest-demo",
    "description": "Demonstrate local file ingestion",
    "global_config": {
        "doc_column": "content",
        "disable_validation": True,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt",

            }}
        }
    ]
}

print("Executing ingest operator...")
manager = DocpipeFlowManager(flow_def=ingest_flow)
manager.execute()

print("\n✓ Ingest completed!")
print("\nKey Features:")
print("  - Loads files from local filesystem")
print("  - Supports file type filtering")
print("  - Parallel processing with max_workers")
print("  - Handles multiple file formats")

## 2. Extract Operators

### ExtractOperator - Extract text from documents using Docling

In [ ]:
extract_flow = {
    "flow_name": "extract-demo",
    "description": "Demonstrate text extraction",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        }
    ]
}

print("Executing extract operator...")
manager = DocpipeFlowManager(flow_def=extract_flow)
manager.execute()

print("\n✓ Extraction completed!")
print("\nKey Features:")
print("  - Extracts text from PDFs, DOCX, and other formats")
print("  - Supports VLM (Vision-Language Models) for enhanced extraction")
print("  - Optional entity extraction with LLMs")
print("  - Preserves document structure and metadata")

## 3. Chunking Operators

### Chunker - Split documents into smaller pieces

In [ ]:
chunking_flow = {
    "flow_name": "chunk-demo",
    "description": "Demonstrate document chunking",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "chunk",
            "type": "chunker",
            "depends_on": ["extract"],
            "config": {
                "chunk_type": "simple",
                "chunk_size": 512,
                "chunk_overlap": 50
            }
        }
    ]
}

print("Executing chunking operator...")
manager = DocpipeFlowManager(flow_def=chunking_flow)
manager.execute()

print("\n✓ Chunking completed!")
print("\nKey Features:")
print("  - Splits documents into manageable chunks")
print("  - Configurable chunk size and overlap")
print("  - Semantic-aware chunking with Docling")
print("  - Preserves context across chunks")

## 4. Quality Operators

### Language Detection - Identify document language

In [ ]:
language_flow = {
    "flow_name": "language-demo",
    "description": "Demonstrate language detection",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "language_detect",
            "type": "lang_detect",
            "depends_on": ["extract"],
            "config": {
                "provider": "fasttext",
                "doc_column": "content"
            }
        }
    ]
}

print("Executing language detection...")
manager = DocpipeFlowManager(flow_def=language_flow)
manager.execute()

print("\n✓ Language detection completed!")

### Readability - Calculate readability scores

In [ ]:
readability_flow = {
    "flow_name": "readability-demo",
    "description": "Demonstrate readability scoring",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "readability",
            "type": "readability",
            "depends_on": ["extract"],
            "config": {
                "readability_score_list": [
                    "flesch_reading_ease",
                    "flesch_kincaid_grade",
                    "gunning_fog",
                    "smog_index",
                    "coleman_liau_index",
                    "automated_readability_index",
                    "dale_chall_readability_score",
                    "difficult_words",
                    "linsear_write_formula",
                    "text_standard",
                    "spache_readability",
                    "mcalpine_eflaw",
                    "reading_time"
                ]
            }
        }
    ]
}

print("Executing readability scoring...")
manager = DocpipeFlowManager(flow_def=readability_flow)
manager.execute()

print("\n✓ Readability scoring completed!")
print("\nAdded metrics: flesch_reading_ease, flesch_kincaid_grade, etc.")

### Deduplication - Remove duplicate documents

In [ ]:
dedup_flow = {
    "flow_name": "dedup-demo",
    "description": "Demonstrate deduplication",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../tests/fixtures/customer_support_docs"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "dedup",
            "type": "ededup",
            "depends_on": ["extract"],
            "config": {
                "doc_column": "content"
            }
        }
    ]
}

print("Executing deduplication...")
print("Using customer support docs with known duplicates:")
print("  - feedback.txt + feedback_duplicate.txt")
print("  - tech_support.txt + tech_support_duplicate.txt")
manager = DocpipeFlowManager(flow_def=dedup_flow)
result = manager.execute()

print("\n✓ Deduplication completed!")
print("\nExpected: 11 documents ingested → 9 documents after dedup (2 duplicates removed)")

## 5. Functional Operators

### Embeddings - Generate vector embeddings

In [ ]:
embeddings_flow = {
    "flow_name": "embeddings-demo",
    "description": "Demonstrate embedding generation",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "embeddings",
            "type": "embeddings",
            "depends_on": ["extract"],
            "config": {
                "provider": "litellm",
                "doc_column": "content",
                "provider_config": {
                    "model_id": "ollama/nomic-embed-text",
                    "api_base": "http://localhost:11434",
                    "api_key": "<YOUR-API-KEY>"
                }
            }
        }
    ]
}

print("Executing embeddings generation...")
print("Note: Requires Ollama running with nomic-embed-text model")
try:
    manager = DocpipeFlowManager(flow_def=embeddings_flow)
    manager.execute()
    print("\n✓ Embeddings generated!")
    print("\nAdded column: embedding (768-dimensional vector)")
except Exception as e:
    print(f"\n✗ Failed: {e}")
    print("Make sure Ollama is running: ollama serve")
    print("And model is available: ollama pull nomic-embed-text")

## 6. Storage Operators

### VectorDB - Store embeddings in vector database

In [ ]:
# Set OpenSearch credentials
import os
from pathlib import Path

# Option 1: Load from .env file (recommended)
# Copy .env.example to .env in the project root and update with your credentials
# Look for .env in project root (two levels up from this notebook)
project_root = Path.cwd()
if (project_root / 'examples' / 'notebooks').exists():
    # We're already in project root
    env_file = project_root / '.env'
else:
    # We're in a subdirectory, go up to find project root
    env_file = project_root.parent.parent / '.env'

if env_file.exists():
    from dotenv import load_dotenv
    load_dotenv(env_file, override=True)
    print(f"✓ Loaded credentials from .env file: {env_file}")
else:
    print(f"⚠ .env file not found at: {env_file}")
    print("  Tip: Copy .env.example to .env in project root and update OPENSEARCH_USERNAME and OPENSEARCH_PASSWORD")

# Option 2: Or set them in your shell before starting Jupyter:
# export OPENSEARCH_USERNAME=your_username
# export OPENSEARCH_PASSWORD=your_password

# Option 3: Set environment variables manually in notebook (if .env not available)
if not os.getenv('OPENSEARCH_USERNAME'):
    os.environ['OPENSEARCH_USERNAME'] = 'admin'
if not os.getenv('OPENSEARCH_PASSWORD'):
    os.environ['OPENSEARCH_PASSWORD'] = '<YOUR-OPENSEARCH-PASSWORD>'

print(f"✓ OpenSearch credentials configured (username: {os.environ.get('OPENSEARCH_USERNAME', 'admin')})")

In [ ]:
import os

vectordb_flow = {
    "flow_name": "vectordb-demo",
    "description": "Demonstrate vector storage",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt"

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "chunk",
            "type": "chunker",
            "depends_on": ["extract"],
            "config": {
                "chunk_type": "simple",
                "chunk_size": 512,
                "chunk_overlap": 50
            }
        },
        {
            "name": "embeddings",
            "type": "embeddings",
            "depends_on": ["chunk"],
            "config": {
                "provider": "litellm",
                "doc_column": "content",
                "provider_config": {
                    "model_id": "ollama/nomic-embed-text",
                    "api_base": "http://localhost:11434",
                    "api_key": "<YOUR-API-KEY>"
                }
            }
        },
        {
            "name": "vectordb",
            "type": "vectordb",
            "depends_on": ["embeddings"],
            "config": {
                "provider": "opensearch",
                "index_name": "demo-index",
                "doc_id_column": "doc_id_hash",
                "embeddings_column": "embeddings",
                "vector_dimension": 768,
                "create_index": True,
                "provider_config": {
                    "host": "localhost",
                    "port": 9200,
                    "username": os.getenv("OPENSEARCH_USERNAME", "admin"),
                    "password": os.getenv("OPENSEARCH_PASSWORD", "admin"),
                    "use_ssl": False,
                    "verify_certs": False,
                    "engine": "faiss",
                    "algorithm": "hnsw"
                },
                "available_features": {
                    "doc_id_hash": {
                        "name": "Document ID",
                        "available_for_vector_db": True,
                        "mandatory_for_vector_db": True,
                        "type": "string",
                        "is_primary": True
                    },
                    "content": {
                        "name": "Content",
                        "available_for_vector_db": True,
                        "type": "string"
                    },
                    "embeddings": {
                        "name": "Embeddings",
                        "available_for_vector_db": True,
                        "mandatory_for_vector_db": True,
                        "type": "vector"
                    }
                }
            }
        }
    ]
}

print("Executing vector storage...")
print("Note: Requires Ollama and OpenSearch running")
try:
    manager = DocpipeFlowManager(flow_def=vectordb_flow)
    manager.execute()
    print("\n✓ Vectors stored in OpenSearch!")
    print("\nIndex: demo-index")
except Exception as e:
    print(f"\n✗ Failed: {e}")
    print("Make sure OpenSearch is running on port 9200")

## Summary

### Operator Categories Demonstrated

1. **Ingest Operators**
   - ✓ IngestLocal - Local filesystem ingestion

2. **Extract Operators**
   - ✓ ExtractOperator - Text and entity extraction
   - Supports Docling, VLM, and LLM-based extraction

3. **Chunking Operators**
   - ✓ Chunker - Simple, semantic, and hybrid chunking

4. **Quality Operators**
   - ✓ Language Detection - 176 language support
   - ✓ Readability - Flesch scores and grade levels
   - ✓ Deduplication - Remove duplicate documents
   - PII/HAP Detection, Redaction, ML Enrichment

5. **Functional Operators**
   - ✓ Embeddings - Vector generation

6. **Storage Operators**
   - ✓ VectorDB - OpenSearch, Milvus

## Next Steps

- **[03_document_extraction.ipynb](03_document_extraction.ipynb)** - Deep dive into extraction
- **[04_embeddings_vectordb.ipynb](04_embeddings_vectordb.ipynb)** - Vector operations
- **[05_quality_operators.ipynb](05_quality_operators.ipynb)** - Quality assessment

## Learn More

- List operators with details: `DocpipeFlowManager.list_operators(verbose=True)`
- Check [Operator Reference](../../docs/OPERATOR_REFERENCE.md) for complete documentation
- See [examples/](../../examples/) for more operator examples